# Configurator Usability Analysis
This notebook guides through the process of the MLLM-based Usability Analysis of Configurator User Interfaces.

This notebook is divided into four main parts:

1.  **Setup and Configuration**: Loads necessary data, including sample descriptions and configurator usability criteria. Additionally, the connection to the LLM is configured.
2.  **Usability Analysis**: For each sample recording, a multimodal LLM is prompted to assess the usability based on each of the provided usability criteria for configurators. The results are saved to a JSON file.

## Setup and Configuration

In [ ]:
import json
import os

DATA_PATH = "./data/"

# Load the sample description file.
with open(os.path.join(DATA_PATH, "sample_description.json")) as f:
    sample_description = json.load(f)

# Load usability criteria. These will be used one by one to guide the LLM in its analysis of the user interface.
with open(os.path.join("configurator_usability_criteria.json")) as f:
    usability_criteria = json.load(f)

In [ ]:
def load_video(video_path):
    """
    Reads a video file from the given path and returns it as a byte stream.
    This is necessary for passing the video data to the multimodal LLM.
    """
    with open(video_path, "rb") as f:
        video = f.read()
    return video

### Configure LLM
This section defines the core components for interacting with the LLM:

*   **API Key and Model**: Credentials and the specific model to be used.
*   **Response Functions**: Helper functions to send requests to the LLM and parse the JSON response.

In [ ]:
# --- IMPORTANT ---
# Replace with your actual Google AI API key.
GENAI_API_KEY = "YOUR_API_KEY_HERE"

# Define the specific generative model to be used for the evaluation.
GENAI_MODEL = "gemini-2.5-flash"

In [ ]:
import json


def repair_json_quotes(json_string: str) -> str:
    """
    Repairs a JSON string by escaping unescaped double quotes within its string values.
    This function iterates through the string character by character to reliably
    determine if a quote is inside a string literal or part of the JSON structure.

    Args:
        json_string: The potentially malformed JSON string.

    Returns:
        A repaired JSON string.
    """
    repaired = []
    in_string = False
    i = 0
    while i < len(json_string):
        char = json_string[i]

        if char == '"':
            # Check if this quote is escaped
            if i > 0 and json_string[i - 1] == '\\':
                # This is an escaped quote, so just append it
                repaired.append(char)
                i += 1
                continue

            # This is an unescaped quote.
            # It can either be a structural quote (toggling in_string)
            # or a quote inside a string that needs to be escaped.
            if not in_string:
                # We are not in a string, so this is an opening structural quote
                in_string = True
                repaired.append(char)
            else:
                # We are inside a string. Is this a closing quote or one to escape?
                # A simple heuristic: if the next non-whitespace character is a comma,
                # closing brace/bracket, or colon, it's likely a structural closing quote.

                # Look ahead for the next non-whitespace character
                next_char_index = i + 1
                while next_char_index < len(json_string) and json_string[next_char_index].isspace():
                    next_char_index += 1

                if next_char_index < len(json_string) and json_string[next_char_index] in [',', '}', ']', ':']:
                    # This looks like a closing structural quote
                    in_string = False
                    repaired.append(char)
                else:
                    # This is a quote inside a string that needs to be escaped
                    repaired.append('\\"')
        else:
            repaired.append(char)

        i += 1

    return "".join(repaired)

In [ ]:
from json import JSONDecodeError
from google import genai
from google.genai.types import Part

def get_response_json(system_prompt, user_prompt, video=None, video_mime_type="video/mp4"):
    """
    Sends a request to the generative AI model and parses the response as JSON.
    This function handles multimodal input (text and video) and is designed for
    structured data extraction from the LLM's response.

    Args:
        system_prompt (str): The system-level instruction for the model.
        user_prompt (str): The user-facing prompt with task details.
        video (bytes, optional): The video data as a byte stream. Defaults to None.
        video_mime_type (str, optional): The MIME type of the video. Defaults to "video/mp4".

    Returns:
        dict or google.generativeai.types.GenerateContentResponse: A dictionary parsed from the JSON
        response. If JSON decoding fails, returns the raw response object.
    """
    client = genai.Client(api_key=GENAI_API_KEY)

    media_content = []
    if video is not None:
        media_content = [Part.from_bytes(data=video, mime_type=video_mime_type)]

    response = client.models.generate_content(
        model=GENAI_MODEL,
        contents=[*media_content, user_prompt],
        config={
            "system_instruction": system_prompt,
            "temperature": 0.0
        }
        )
    try:
        response_json = json.loads(repair_json_quotes(response.candidates[0].content.parts[0].text.lstrip("```json").rstrip("```")))
    except JSONDecodeError:
        print(f"Error decoding JSON. Return the raw response.")
        return response.candidates[0].content.parts[0].text
    return response_json

def get_response_text(user_prompt):
    """
    Sends a text-only request to the generative AI model and returns the response as a string.
    This function is used for simpler, text-based summarization tasks.

    Args:
        user_prompt (str): The user-facing prompt.

    Returns:
        str: The text content of the model's response.
    """
    client = genai.Client(api_key=GENAI_API_KEY)
    response = client.models.generate_content(
        model=GENAI_MODEL,
        contents=[user_prompt],
        config={
            "temperature": 0.0
        }
    )
    return response.candidates[0].content.parts[0].text

## Usability Analysis
This section executes the core logic of the analysis.

The `run_usability_analysisi` function iterates through each samplea and, for each, it loads the corresponding screen recording and then loops through all configurator usability criteria.

For each criterion, it:
1.  Constructs a detailed prompt.
2.  Sends the prompt and the video to the multimodal LLM.
3.  Appends the structured JSON response to a list.

Finally, the collected evaluations are saved to a timestamped JSON file in the `reports` directory.

In [ ]:
# The system prompt sets the overall context for the LLM.
# It instructs the model to act as a usability expert.
ANALYSIS_SYSTEM_PROMPT = f"""
You are a highly experienced usability expert.
Your task is to analyze the user interface of configurators.
Use the provided recorded user interaction to support a detailed, criteria-based analysis.
Assess each criterion independently.
Base your analysis strictly on the provided material and context - do not make assumptions.
Do not mention issues if these are not visible in the provided screen recording.
Tailor your description and recommendations to the configurator context including the product domain.
Provide structured findings with a clear issue description and and actionable improvement recommendation.
"""

In [ ]:
def get_analysis_user_prompt(criterion_name, criterion_description):
    """
    Constructs the detailed user prompt for the LLM-based usability evaluation.

    Args:
        criterion_name (str): The name of the criterion to be analyzed.
        criterion_description (str): A description of the criterion.

    Returns:
        str: The fully formatted prompt to be sent to the LLM.
    """

    prompt = ""

    prompt += f"""
Criterion Under Evaluation:
{criterion_name}

Criterion Description:
{criterion_description}

Analysis Instructions:
Analyze the user interface for the criterion listed above. Follow these steps:
1. Identify usability issues related to the criterion.
2. Indicate the severity of usability issues related to the criterion using the following scale:
   - No Issue
   - Minor Issue
   - Major Issue
If you identified issues:
3. Explain the issue in detail without restating the criterion description.
4. Provide a practical, actionable recommendation to fix the issue tailored to the configurator.

Response Guideline:
Keep each issue’s analysis and recommendation concise (1–5 sentences).

Format your response as JSON as follows:
{{
  "criterion": "{criterion_name}",
  "severity": "[No Issue | Minor Issue | Major Issue]"
  "description": "[Brief description of the usability issue (optional)]",
  "recommendation": "[Actionable suggestion for improvement (optional)]",
}}

Answer with JSON only.
"""

    return prompt

In [ ]:
REPORT_FILE = f"analysis_report.json"

In [ ]:
from pathlib import Path

def already_processed(results_path, app_id, criterion_id):
    """Check if a sheet already processed."""
    if not Path(results_path).exists():
        return False
    with open(results_path, "r") as f:
        for line in f:
            rec = json.loads(line)
            if rec["app_id"] == app_id and rec["criterion_id"] == criterion_id:
                return True
    return False

def safe_append_jsonl(path, record):
    """Append record to JSONL safely."""
    with open(path, "a") as f:
        f.write(json.dumps(record) + "\n")

In [ ]:
import time

def run_usability_analysis():
    """
    Orchestrates the usability analysis process.
    It iterates through each sample defined in the sample_description.json file.
    For each sample, it performs a criterion-based analysis using the associated video recording.
    """
    for application in sample_description["applications"]:
        encoded_video = load_video(os.path.join(DATA_PATH, application["video_file"]))
        for criterion_id, criterion in usability_criteria.items():
            if already_processed(REPORT_FILE, application["app_id"], criterion_id):
                print(f"Skip {application['app_id']} for criterion {criterion_id}")
                continue

            print(f"Processing {application['app_id']} for criterion {criterion_id}")
            try:
                user_prompt = get_analysis_user_prompt(criterion["name"], criterion["description"])
                start_time = time.time()
                response_json = get_response_json(ANALYSIS_SYSTEM_PROMPT, user_prompt, video=encoded_video, video_mime_type="video/mp4")
                end_time = time.time()

                response_json["app_id"] = application['app_id']
                response_json["criterion_id"] = criterion_id
                response_json["request_duration"] = round(end_time - start_time, 4)
                safe_append_jsonl(REPORT_FILE, response_json)
            except Exception as e:
                print(f"Error processing {application['app_id']} for criterion {criterion_id}: {e}")

In [ ]:
# This cell triggers the entire analysis process.
# Note: This can take a longer time to run, as it makes multiple calls to the LLM API.

print("Starting LLM-based usability analysis...")
run_usability_analysis()

print("Analysis complete and report stored.")